# Cross-Country Comparison and Climate Vulnerability Ranking (Task 3)

This notebook compares Ethiopia, Kenya, Sudan, Tanzania, and Nigeria to identify relative climate vulnerability and support COP32 positioning.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import kruskal, f_oneway

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [ ]:
DATA_DIR = Path('..') / 'data'
COUNTRIES = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']

def clean_country(raw_path: Path, clean_path: Path, country_name: str) -> pd.DataFrame:
    df = pd.read_csv(raw_path)
    df['Country'] = country_name.title()
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j', errors='coerce')
    df['Month'] = df['Date'].dt.month
    df = df.replace(-999, np.nan)
    df = df.drop_duplicates().copy()

    row_missing_pct = df.isna().mean(axis=1)
    df = df[row_missing_pct <= 0.30].copy()
    weather_fill_cols = [
        c for c in ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']
        if c in df.columns
    ]
    df = df.sort_values('Date').reset_index(drop=True)
    df[weather_fill_cols] = df[weather_fill_cols].ffill()
    clean_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(clean_path, index=False)
    return df

dfs = []
for country in COUNTRIES:
    clean_path = DATA_DIR / f'{country}_clean.csv'
    raw_path = DATA_DIR / f'{country}.csv'
    if clean_path.exists():
        df_country = pd.read_csv(clean_path)
        if 'Date' not in df_country.columns:
            df_country['Date'] = pd.to_datetime(df_country['YEAR'] * 1000 + df_country['DOY'], format='%Y%j', errors='coerce')
        else:
            df_country['Date'] = pd.to_datetime(df_country['Date'], errors='coerce')
        if 'Country' not in df_country.columns:
            df_country['Country'] = country.title()
        if 'Month' not in df_country.columns:
            df_country['Month'] = df_country['Date'].dt.month
    else:
        df_country = clean_country(raw_path, clean_path, country)
    dfs.append(df_country)

df_all = pd.concat(dfs, ignore_index=True)
df_all['Country'] = df_all['Country'].str.title()
df_all = df_all.sort_values(['Country', 'Date']).reset_index(drop=True)
print('Combined shape:', df_all.shape)
display(df_all.head())


## Temperature Trend Comparison

In [ ]:
monthly_t2m = (
    df_all.resample('ME', on='Date')
    .agg(T2M=('T2M', 'mean'), Country=('Country', 'first'))
    .reset_index()
)

# The above approach with first country after resample is not grouped, so regroup directly:
monthly_t2m = (
    df_all.groupby(['Country', pd.Grouper(key='Date', freq='ME')])['T2M']
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_t2m, x='Date', y='T2M', hue='Country')
plt.title('Monthly Average T2M by Country (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Temperature (degC)')
plt.legend(title='Country')
plt.tight_layout()
plt.show()

t2m_summary = (
    df_all.groupby('Country')['T2M']
    .agg(['mean', 'median', 'std'])
    .round(2)
    .sort_values('mean', ascending=False)
)
display(t2m_summary)


## Precipitation Variability Comparison

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_all, x='Country', y='PRECTOTCORR')
plt.title('Daily PRECTOTCORR Distribution by Country')
plt.xlabel('Country')
plt.ylabel('Precipitation (mm/day)')
plt.tight_layout()
plt.show()

prec_summary = (
    df_all.groupby('Country')['PRECTOTCORR']
    .agg(['mean', 'median', 'std'])
    .round(2)
    .sort_values('std', ascending=False)
)
display(prec_summary)


## Extreme Event Frequency

In [ ]:
extreme_heat = (
    df_all.assign(Year=df_all['Date'].dt.year, ExtremeHeat=(df_all['T2M_MAX'] > 35).astype(int))
    .groupby(['Country', 'Year'])['ExtremeHeat']
    .sum()
    .reset_index(name='ExtremeHeatDays')
)

def consecutive_dry_days(group: pd.DataFrame) -> pd.Series:
    dry = (group['PRECTOTCORR'] < 1).astype(int)
    max_run = 0
    current = 0
    for v in dry:
        if v == 1:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return pd.Series({'MaxConsecutiveDryDays': max_run, 'TotalDryDays': int(dry.sum())})

dry_stats = (
    df_all.assign(Year=df_all['Date'].dt.year)
    .sort_values('Date')
    .groupby(['Country', 'Year'], group_keys=False)
    .apply(consecutive_dry_days)
    .reset_index()
)

heat_avg = extreme_heat.groupby('Country')['ExtremeHeatDays'].mean().reset_index()
dry_avg = dry_stats.groupby('Country')['MaxConsecutiveDryDays'].mean().reset_index()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=heat_avg.sort_values('ExtremeHeatDays', ascending=False), x='Country', y='ExtremeHeatDays', ax=ax[0])
ax[0].set_title('Average Annual Extreme Heat Days (T2M_MAX > 35C)')
ax[0].set_ylabel('Days per Year')

sns.barplot(data=dry_avg.sort_values('MaxConsecutiveDryDays', ascending=False), x='Country', y='MaxConsecutiveDryDays', ax=ax[1])
ax[1].set_title('Average Annual Maximum Consecutive Dry Days (PRECTOTCORR < 1 mm)')
ax[1].set_ylabel('Days')
plt.tight_layout()
plt.show()

display(heat_avg.sort_values('ExtremeHeatDays', ascending=False))
display(dry_avg.sort_values('MaxConsecutiveDryDays', ascending=False))


## Statistical Testing (ANOVA and Kruskal-Wallis)

In [ ]:
samples = [df_all[df_all['Country'] == c]['T2M'].dropna().values for c in sorted(df_all['Country'].unique())]
anova_stat, anova_p = f_oneway(*samples)
kw_stat, kw_p = kruskal(*samples)

stats_df = pd.DataFrame(
    {
        'test': ['ANOVA', 'Kruskal-Wallis'],
        'statistic': [anova_stat, kw_stat],
        'p_value': [anova_p, kw_p]
    }
).round(6)
display(stats_df)

if anova_p < 0.05:
    print('ANOVA suggests statistically significant differences in T2M across countries.')
else:
    print('ANOVA does not show statistically significant T2M differences at 5% level.')


## Vulnerability Ranking

In [ ]:
def warming_slope(group: pd.DataFrame) -> float:
    monthly = (
        group.groupby(pd.Grouper(key='Date', freq='ME'))['T2M']
        .mean()
        .dropna()
        .reset_index()
    )
    if len(monthly) < 2:
        return np.nan
    x = np.arange(len(monthly))
    y = monthly['T2M'].values
    slope = np.polyfit(x, y, 1)[0]
    return float(slope)

warming = df_all.groupby('Country').apply(warming_slope).reset_index(name='warming_slope_per_month')
prec_var = df_all.groupby('Country')['PRECTOTCORR'].std().reset_index(name='precip_std')
heat_metric = heat_avg.rename(columns={'ExtremeHeatDays': 'avg_extreme_heat_days'})
dry_metric = dry_avg.rename(columns={'MaxConsecutiveDryDays': 'avg_max_consecutive_dry_days'})

rank_df = warming.merge(prec_var, on='Country').merge(heat_metric, on='Country').merge(dry_metric, on='Country')

for col in ['warming_slope_per_month', 'precip_std', 'avg_extreme_heat_days', 'avg_max_consecutive_dry_days']:
    rank_df[col + '_rank'] = rank_df[col].rank(method='min', ascending=False)

rank_df['vulnerability_score'] = rank_df[[
    'warming_slope_per_month_rank',
    'precip_std_rank',
    'avg_extreme_heat_days_rank',
    'avg_max_consecutive_dry_days_rank'
]].sum(axis=1)

rank_df = rank_df.sort_values('vulnerability_score', ascending=False).reset_index(drop=True)
rank_df['vulnerability_rank'] = np.arange(1, len(rank_df) + 1)
display(rank_df)


## COP32 Framing (5 Key Points)

- **Warming trajectory:** Use `warming_slope_per_month` to identify which country is warming fastest and report whether warming appears persistent or volatile by season.
- **Precipitation instability:** Use `precip_std` and precipitation boxplots to identify the country with the most unstable rainfall profile.
- **Heat and drought stress:** Use `avg_extreme_heat_days` and `avg_max_consecutive_dry_days` to frame compound climate stress exposure.
- **Ethiopia's comparative profile:** Compare Ethiopia directly against neighboring countries on all four ranking dimensions.
- **Priority climate finance ask:** Champion the top-ranked vulnerable country (or tied group) for adaptation finance, early warning systems, climate-resilient agriculture, and drought-risk management.
